# Example 01 Lecture Note: Build a GradedComplex by Hand

This notebook strips away geometry and works directly with the algebraic object.

## Learning goals
- Build a tiny graded complex manually (cells, boundaries, grades).
- Understand "graded by filtration parameters" vs "degree in cohomology".
- Encode to `H^t` and inspect module-level summaries.
- Export one deterministic feature row.
        


## Conceptual distinction used throughout

- `GradedComplex`: each cell has a multi-grade (filtration coordinates).
- `degree=t` in `encode(...)`: choose cohomology degree `H^t` to return.

So in this notebook:
1) we hand-enter filtration grading data,
2) then ask for `H^0` over the induced finite poset.
        


In [ ]:
using Random
using SparseArrays
using Statistics
using Printf

repo_root = if isdir(joinpath(pwd(), "src"))
    pwd()
elseif isdir(joinpath(pwd(), "..", "src"))
    normpath(joinpath(pwd(), ".."))
else
    pwd()
end

try
    using PosetModules
catch
    include(joinpath(repo_root, "src", "PosetModules.jl"))
    using .PosetModules
end
const PM = PosetModules

outdir = joinpath(repo_root, "examples", "_outputs", "01_graded_complex_notebook")
mkpath(outdir)
println("repo_root = ", repo_root)
println("outdir    = ", outdir)


## Step 1: Define a tiny chain complex

Cells:
- Dimension 0: `v1, v2, v3`
- Dimension 1: `e1, e2`

Boundary map `d1: C1 -> C0` is entered explicitly as a sparse matrix.
        


In [ ]:
cells_by_dim = [Int[1, 2, 3], Int[1, 2]]

B1 = sparse(
    [1, 2, 2, 3],
    [1, 1, 2, 2],
    [1, -1, 1, -1],
    3,
    2,
)

println("cell counts by dim = ", map(length, cells_by_dim))
println("B1 size            = ", size(B1))


## Step 2: Attach 2-parameter grades to every cell

Grade order must follow dimension-major cell order:
`[v1, v2, v3, e1, e2]`.

This explicit order is the main source of mistakes in hand-built examples.
        


In [ ]:
grades = [
    [0.0, 0.0],
    [0.6, 0.0],
    [1.2, 0.0],
    [0.6, 0.4],
    [1.2, 0.8],
]

G = PM.GradedComplex(cells_by_dim, [B1], grades)
println("G type = ", typeof(G))


## Step 3: Choose explicit axes and encode

We choose explicit axes so the poset discretization is fully visible.

Alternative: use `axes_policy=:encoding` for adaptive axes derived from data.
        


In [ ]:
axes = (
    collect(range(0.0, stop=1.6, length=5)),
    collect(range(0.0, stop=1.2, length=4)),
)

spec = PM.FiltrationSpec(; kind=:graded, axes=axes)
enc = PM.encode(G, spec; degree=0, field=PM.CoreModules.F2(), cache=:auto)

println("spec type = ", typeof(spec))
println("enc type  = ", typeof(enc))
println("enc.M type= ", typeof(enc.M))
println("enc.pi    = ", typeof(enc.pi))
println("|P|       = ", PM.nvertices(enc.P))


## Step 4: Invariant checks on the encoded module

We keep `axes_policy=:as_given` because axes were explicitly chosen above.
        


In [ ]:
opts = PM.InvariantOptions(
    ;
    axes=axes,
    axes_policy=:as_given,
    max_axis_len=64,
    threads=false,
    strict=true,
    pl_mode=:fast,
)

rh = PM.restricted_hilbert(enc.M)
rank_tbl = PM.rank_invariant(enc; opts=opts, store_zeros=true)

println("length(restricted_hilbert) = ", length(rh))
println("rank table entries         = ", length(rank_tbl))


## Step 5: Build a deterministic feature row

`RankGridSpec` is used because feature dimension is explicit and easy to audit.
        


In [ ]:
rank_spec = PM.RankGridSpec(nvertices=PM.nvertices(enc.P), store_zeros=true, threads=false)

fs = PM.batch_transform(
    [enc],
    rank_spec;
    opts=opts,
    batch=PM.BatchOptions(threaded=false, backend=:threads, progress=false),
    cache=:auto,
    idfun=_ -> "graded_hand_001",
)

println("nfeatures(rank_spec) = ", PM.nfeatures(rank_spec))
println("feature matrix shape = ", size(fs.X))


## Step 6: Export outputs
        


In [ ]:
wide_path = joinpath(outdir, "graded_hand__wide.csv")
long_path = joinpath(outdir, "graded_hand__long.csv")

PM.save_features(wide_path, fs; format=:csv, mode=:wide, metadata=true)
PM.save_features(long_path, fs; format=:csv, mode=:long, metadata=true)

g_path = joinpath(outdir, "graded_complex.json")
PM.save_dataset_json(g_path, G)

println("wide csv    = ", wide_path)
println("long csv    = ", long_path)
println("graded json = ", g_path)


## Discussion prompts
- Which mistakes in grade ordering would still produce a valid object but wrong semantics?
- How would switching from `F2()` to `QQ()` change interpretation of boundary signs?
- What changes if you ask for `degree=1` instead of `degree=0`?
        
